# Datatype Handling

GeoKit uses GDAL internally to perform many of the provided functions and methods. The values of rasters and fields in vector datasets are stored as C data types with specific bit lengths. Using smaller data types with fewer bits can improve performance and reduce memory usage. However, choosing a data type with an insufficient bit length, or the wrong data type, can result in so-called overflow errors for both [integer](https://en.wikipedia.org/wiki/Integer_overflow) and [float](https://en.wikipedia.org/wiki/Floating-point_arithmetic#Range_of_floating-point_numbers) data types.

In Python, conversion is usually done automatically. GeoKit also has internal logic to select a safe data type, which prevents these overflow errors. To inspect the logic, use the methods shown in this example.

## Get the datatype for individual numbers 

To obtain a minimum required datatype for an individual number use the "get_valid_gdal_data_type_as_string" method

In [1]:
from geokit.c_data_type_handler import MinimumCDataTypeHandler
import numpy as np

# To showcase the data type detection based on single numbers
for current_number in [
    5,
    0 - 5,
    230,
    5000,
    60000,
    9223372036854775807,
    18446744073709551615,
    True,
    False,
    1 / 3,
    -1 / 3,
    np.nan,
    np.inf,
    -np.inf,
    1.0,
    3 * 10**37,
    3 * 10**40,
]:
    data_type=MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(list_of_numbers=[current_number])
    print(f"Number: {current_number} -> Data type: {data_type}")


Number: 5 -> Data type: GDT_Int8
Number: -5 -> Data type: GDT_Int8
Number: 230 -> Data type: GDT_Byte
Number: 5000 -> Data type: GDT_Int16
Number: 60000 -> Data type: GDT_UInt16
Number: 9223372036854775807 -> Data type: GDT_Int64
Number: 18446744073709551615 -> Data type: GDT_UInt64
Number: True -> Data type: GDT_Int8
Number: False -> Data type: GDT_Int8
Number: 0.3333333333333333 -> Data type: GDT_Float32
Number: -0.3333333333333333 -> Data type: GDT_Float32
Number: nan -> Data type: GDT_Float32
Number: inf -> Data type: GDT_Float32
Number: -inf -> Data type: GDT_Float32
Number: 1.0 -> Data type: GDT_Int8
Number: 30000000000000000000000000000000000000 -> Data type: GDT_Float32
Number: 30000000000000000000000000000000000000000 -> Data type: GDT_Float64


## Use multiple numbers

If you have several numbers of the same type that you want to store, you can simply add them to the list of numbers argument.

In [2]:
data_type=MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(list_of_numbers=[9,300])
print(f"common datatype: {[9, 300]} -> Data type: {data_type}")

common datatype: [9, 300] -> Data type: GDT_Int16


## All Supported Datatypes

GeoKit does not support all C data types, nor all C data types supported by GDAL, because some are rarely found in real-world geodata. Below the supported data is shown:

In [3]:
from geokit.data_types import _gdal_c_raster_data_types_list,_gdal_c_raster_data_types_abbreviations_list

print("Supported c data types in gdal:\n", _gdal_c_raster_data_types_list)
print("Aliases for Supported c data types in gdal:\n", _gdal_c_raster_data_types_abbreviations_list)

Supported c data types in gdal:
 ['GDT_Int8', 'GDT_Byte', 'GDT_UInt16', 'GDT_Int16', 'GDT_UInt32', 'GDT_Int32', 'GDT_UInt64', 'GDT_Int64', 'GDT_Float32', 'GDT_Float64']
Aliases for Supported c data types in gdal:
 ['Int8', 'int8', 'Byte', 'byte', 'UInt16', 'uint16', 'Int16', 'int16', 'UInt32', 'uint32', 'Int32', 'int32', 'UInt64', 'uint64', 'Int64', 'int64', 'Float32', 'float32', 'Float64', 'float64']


## Set Minimum Datatype

In some cases, you may need to specify a particular data type, for example, to ensure compatibility with other software. The 'get_valid_gdal_data_type_as_string' function allows you to specify this minimum data type. However, if the numbers cannot be stored in this data type, the smallest suitable data type is returned. For automation purposes, multiple minimum data types can also be passed.

In [4]:
# Show the minimum datatype to store a 7
numbers_to_inspect=[7]
minimum_gdal_type_list=None # no minimum data type should be considered
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect, minimum_gdal_type_list=minimum_gdal_type_list
)
print(f"Numbers inspected: {numbers_to_inspect}, minimum data type defined: {minimum_gdal_type_list}, -> Data type: {data_type}")

# Show the minimum datatype to store a 7 with a minimum data type of GDT_UInt16
numbers_to_inspect = [7]
minimum_gdal_type_list = ["GDT_Int16"]
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect, minimum_gdal_type_list=minimum_gdal_type_list
)
print(
    f"Numbers inspected: {numbers_to_inspect}, minimum data type defined: {minimum_gdal_type_list}, -> Data type: {data_type}"
)

# Show the minimum datatype to store a 300 with a minimum data type of GDT_Int8
numbers_to_inspect = [300]
minimum_gdal_type_list = ["GDT_Int8"]
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect, minimum_gdal_type_list=minimum_gdal_type_list
)
print(
    f"Numbers inspected: {numbers_to_inspect}, minimum data type defined: {minimum_gdal_type_list}, -> Data type: {data_type}, GDT_Int8 cannot store a 300 thus GDT_Int16 is returned"
)

Numbers inspected: [7], minimum data type defined: None, -> Data type: GDT_Int8
Numbers inspected: [7], minimum data type defined: ['GDT_Int16'], -> Data type: GDT_Int16
Numbers inspected: [300], minimum data type defined: ['GDT_Int8'], -> Data type: GDT_Int16, GDT_Int8 cannot store a 300 thus GDT_Int16 is returned


## Ambiguity due to signed and unsigned integer

Some values can be stored as either a signed or unsigned integer. For example, 30,000 can be stored as an Int16 or a Uint16. In case of ambiguity, GeoKit chooses the signed integer. 

In [5]:
numbers_to_inspect = [30000]
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect,
)
print(
    f"Numbers inspected: {numbers_to_inspect}, -> Data type: {data_type}"
)
numbers_to_inspect = [30000]
minimum_gdal_type_list = ["GDT_UInt16"]
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect, minimum_gdal_type_list=minimum_gdal_type_list
)
print(
    f"Numbers inspected: {numbers_to_inspect}, minimum data type defined: {minimum_gdal_type_list}, -> Data type: {data_type}"
)

Numbers inspected: [30000], -> Data type: GDT_Int16
Numbers inspected: [30000], minimum data type defined: ['GDT_UInt16'], -> Data type: GDT_Int16


## Choose Unsigned Integer over Signed Integer in Ambiguous cases

In order to choose an unsigned integer over a signed one pass the desired data type "user_defined_minimum_gdal_type" 

In [7]:
numbers_to_inspect = [300]
user_defined_minimum_gdal_type = "GDT_UInt16"
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect, user_defined_minimum_gdal_type=user_defined_minimum_gdal_type
)
print(
    f"Numbers inspected: {numbers_to_inspect}, user defined: {user_defined_minimum_gdal_type}, -> Data type: {data_type}"
)


Numbers inspected: [300], user defined: GDT_UInt16, -> Data type: GDT_UInt16


## Detect Unintended Data Conversions

If you require a specific output data type you can use user_defined_minimum_gdal_type argument. It prompts a warning in case another data type has been deemed necessary.

In [8]:
numbers_to_inspect = [-60]
user_defined_minimum_gdal_type = "GDT_UInt16"
data_type = MinimumCDataTypeHandler.get_valid_gdal_data_type_as_string(
    list_of_numbers=numbers_to_inspect, user_defined_minimum_gdal_type=user_defined_minimum_gdal_type
)
print(
    f"Numbers inspected: {numbers_to_inspect}, user defined: {user_defined_minimum_gdal_type}, -> Data type: {data_type}"
)


Numbers inspected: [-60], user defined: GDT_UInt16, -> Data type: GDT_Int8


/fast/home/j-belina/geokit/geokit/c_data_type_handler.py:398: UserWarning: The user-defined minimum GDAL data type: GDT_UInt16, which is understood as the rigorous GDAL datatype: GDT_UInt16, differs from automatically determined data type: GDT_Int8. To silence this warning check the configuration of your function call for configurations that might cause an unintentional overflow error. Otherwise you can just set the data type to the automatically determined data type or to the Python object None.
  warnings.warn(
